# Création du modèle IA + entrainement + visualisation des performance

## 1) import

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from stable_baselines3 import PPO
import matplotlib.pyplot as plt

## 2) 🧠 ForexEnv — Environnement d’apprentissage pour OrionTrader

Ce fichier décrit la logique du code Python `ForexEnv`, qui simule un petit marché de change (Forex) simplifié.  
Cet environnement est compatible avec **Gymnasium / OpenAI Gym**, ce qui permet d’y entraîner des agents de **reinforcement learning** (comme PPO, DQN, etc.).

### 🧩 Description des composants

#### Initialisation (`__init__`)

- `balance`: capital initial du trader (1000 unités)
- `price`: prix actuel du marché (1.1 EUR/USD)
- `action_space`: trois actions possibles
    - 0 → Acheter
    - 1 → Vendre
    - 2 → Attendre
- `observation_space`: observation unique = prix actuel, bornée entre 0 et 10

#### Réinitialisation (`reset`)

- Remet à zéro l’environnement (capital et prix)
- Retourne l’état initial (ici, juste le prix)
- Utilisé automatiquement par l’agent au début de chaque épisode

#### Évolution d’un pas (`step`)

- `step()` est appelée à chaque action de l’agent.
- Le prix évolue de manière aléatoire (marché simulé avec une petite volatilité de ±1%).

#### Gestion des actions

|Action|Description|Formule de la récompense|
|---|---|---|
|0|Buy : on achète du dollar|(self.price - old_price) * 1000|
|1|Sell : on vend du dollar|(old_price - self.price) * 1000|
|2|Hold : on attend|-0.1 (pénalité d’inaction)|

#### ⚙️ Objectif pour l’agent IA

L’agent (par ex. un modèle PPO) doit apprendre à :

- acheter quand le prix va monter,
- vendre quand le prix va baisser,
- et éviter les pertes (ou les longues périodes d’inaction).

Chaque épisode correspond à une session de trading où il essaie de maximiser sa balance finale.

#### 🚀 Prochaines améliorations possibles
- Ajouter un spread (écart entre prix achat/vente)
- Intégrer un historique de prix (observations multi-dimensionnelles)
- Simuler un effet de levier
- Ajouter une mémoire d’état pour permettre des stratégies plus complexes
- Connecter aux vraies données MT5

In [ ]:
class ForexEnv(gym.Env):
    def __init__(self):
        super(ForexEnv, self).__init__()
        self.balance = 1000
        self.price = 1.1
        self.action_space = spaces.Discrete(3)  # 0: Buy, 1: Sell, 2: Hold
        self.observation_space = spaces.Box(low=0, high=10, shape=(1,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        self.balance = 1000
        self.price = 1.1
        return np.array([self.price], dtype=np.float32), {}

    def step(self, action):
        reward = 0
        old_price = self.price
        self.price += np.random.normal(0, 0.01)  # simulation aléatoire du marché

        if action == 0:  # Buy
            reward = (self.price - old_price) * 1000
        elif action == 1:  # Sell
            reward = (old_price - self.price) * 1000
        elif action == 2:  # Hold
            reward = -0.1  # léger coût d'opportunité

        self.balance += reward
        done = self.balance <= 0 or self.balance >= 2000
        return np.array([self.price], dtype=np.float32), reward, done, False, {}

env = ForexEnv()

In [ ]:
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=100_000)
model.save("orion_ppo_model")

In [ ]:
obs, _ = env.reset()
for i in range(20):
    action, _states = model.predict(obs)
    obs, reward, done, truncated, info = env.step(action)
    print(f"Step {i}: Action={action}, Reward={reward:.2f}, Balance={env.balance:.2f}")
    if done:
        print("Episode terminé.")
        break

In [ ]:
prices = []
rewards = []
obs, _ = env.reset()
for _ in range(200):
    prices.append(env.price)
    action, _ = model.predict(obs)
    obs, reward, done, _, _ = env.step(action)
    rewards.append(reward)
    if done:
        break

plt.figure(figsize=(10, 4))
plt.plot(prices, label="EUR/USD")
plt.title("Simulation de prix")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(np.cumsum(rewards))
plt.title("Cumul des récompenses")
plt.show()
